# SuperMAG Processing

This script processes SuperMAG large download cdf files, extracting the relevant station and horizontal magnetic field data in geographic coordinates. The total number of stations is determined, with a key file created to map the station code to geographic coordinates. The data itself is padded to the the total number of stations to allow for consistent array shapes. The data is then saved in a csv file for each geomagnetic field component.

In [1]:
import netCDF4 as nc
import datetime as dt
import numpy as np
import pandas as pd


def get_time(year, month, day, hour, minute):
    """
    Function to convert year, month, day, hour, and minute to datetime object.
    """
    return dt.datetime(year, month, day, hour, minute)


def write_data(fnm, year, dest="."):
    """
    Function to read in cdf files and create csv files for dbe and dbn data. Addtionally, create stns.csv to contain all information about stations.

    Parameters:
        fnm: str
            File path of cdf file
        year: int
            Year of the data
        dest: str (default is '.')
            Destination folder to save the csv files

    Returns:
        None
    """

    # read in cdf files and create csv files
    cdf = nc.Dataset(fnm)
    ids = cdf["id"][0]
    glat = cdf["glat"][0]
    glon = cdf["glon"][0]
    dbe_geo = np.array(cdf["dbe_geo"][:].data)
    dbn_geo = np.array(cdf["dbn_geo"][:].data)
    dbz_geo = np.array(cdf["dbz_geo"][:].data)
    cdf_time = map(
        get_time,
        cdf["time_yr"][:].data,
        cdf["time_mo"][:].data,
        cdf["time_dy"][:].data,
        cdf["time_hr"][:].data,
        cdf["time_mt"][:].data,
    )
    times = list(cdf_time)
    data_dbe = pd.DataFrame(data=dbe_geo, index=times, columns=ids)
    data_dbn = pd.DataFrame(data=dbn_geo, index=times, columns=ids)
    data_dbz = pd.DataFrame(data=dbz_geo, index=times, columns=ids)
    data_dbe.to_csv(dest + f"dbe_geo_{year}.csv", index_label="time", header=True)
    data_dbn.to_csv(dest + f"dbn_geo_{year}.csv", index_label="time", header=True)
    data_dbz.to_csv(dest + f"dbz_geo_{year}.csv", index_label="time", header=True)

    # create stns.csv to contain all information about stations
    ids_df = pd.DataFrame({"stns": ids, "glat": glat, "glon": glon})
    try:
        stns_df = pd.read_csv(dest + "stns.csv")
        stns_df = pd.concat([stns_df, ids_df])
        stns_df.drop_duplicates(subset="stns", inplace=True)
        stns_df.sort_values(by="stns", inplace=True)
        stns_df.to_csv(dest + "stns.csv", index=False)
    except:
        ids_df.sort_values(by="stns", inplace=True)
        ids_df.to_csv(dest + "stns.csv", index=False)

    return

In [2]:
for year in range(2001, 2025):
    print("Processing", year)
    filename = f"/mnt/disks/data/supermag/all_stations_all{year}.netcdf"
    write_data(filename, year, dest="/mnt/disks/data/")

Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
Processing 2023
Processing 2024


In [3]:
stns_all = pd.read_csv("/mnt/disks/data/stns.csv")

In [4]:
def sort_supermag(fnm, stns_all, dest="."):
    """
    Function to sort supermag data and add missing columns.

    Parameters:
        fnm: str
            File path of supermag data
        stns_all: pandas DataFrame
            DataFrame containing all information about stations
        dest: str (default is '.')
            Destination folder to save the csv files
    """

    data = pd.read_csv(dest + fnm, parse_dates=True, index_col="time")
    cols = data.columns
    new_cols = []
    for col in stns_all["stns"]:
        if col not in cols:
            new_cols.append(col)
    ext_data = pd.DataFrame(columns=new_cols, index=data.index)
    data_all = pd.concat([data, ext_data], axis=1)
    data_sorted = data_all.reindex(sorted(data_all.columns), axis=1)
    data_sorted.to_csv(dest + "formatted_" + fnm, index_label="time", header=True)

In [5]:
for year in range(2001, 2025):
    print("Processing", year)
    filename = f"dbe_geo_{year}.csv"
    sort_supermag(filename, stns_all, dest="/mnt/disks/data/")
    filename = f"dbn_geo_{year}.csv"
    sort_supermag(filename, stns_all, dest="/mnt/disks/data/")
    filename = f"dbz_geo_{year}.csv"
    sort_supermag(filename, stns_all, dest="/mnt/disks/data/")

Processing 2001
Processing 2002
Processing 2003
Processing 2004
Processing 2005
Processing 2006
Processing 2007
Processing 2008
Processing 2009
Processing 2010
Processing 2011
Processing 2012
Processing 2013
Processing 2014
Processing 2015
Processing 2016
Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022
Processing 2023
Processing 2024
